# Lab 05 -- Instruction-Based Prompting Deep Dive

**Week 3 - Prompt Engineering and Task-to-Prompt Mapping**

**Focus.** Write crisp, testable instruction prompts. You will pin down a **role**, **constraints**, a **step list**, and an explicit **output contract**, then prove the output is correct with **acceptance tests** you run in Python. You will also control **style and tone** without breaking correctness.

### Outcomes
By the end you can:
1. Encode an output contract as a Pydantic model that rejects malformed model output.
2. Lint a prompt for required structure before it ships.
3. Author acceptance tests that run offline against model output.
4. Show that a weak prompt fails those tests and a disciplined prompt passes them.
5. Vary style and tone while holding the contract fixed.

### How this lab runs
Everything runs **offline and deterministically**. A provided helper, `simulate_model`, stands in for a hosted LLM. Its decisions are always correct; what changes with prompt quality is how well the **output conforms to your contract**, which mirrors how real models drift when instructions are vague. No API key, no network, no cost. An optional appendix shows how to swap in a real model.


### How the self-check works
Each task is graded by a soft checker. Calling `check(name, condition)` records a `PASS` or `FAIL` and never halts the notebook, so a `Run All` on the unfinished notebook completes and simply reports a low score. `run_check(name, fn, expect_error=...)` calls your code inside a guard so an unfinished stub reports `FAIL` instead of crashing the run.

**Setup.** Keep `lab_kit.py` in the same folder as this notebook, and start Jupyter from that folder. Fill in each cell marked `# TODO`, re-run, and drive the final score to green.


In [ ]:
%pip install -r requirements.txt

In [ ]:
import sys, pathlib, platform
sys.path.insert(0, str(pathlib.Path.cwd()))   # make lab_kit importable

from lab_kit import (
    REQUIRED_SECTIONS, render_prompt, simulate_model, Grader,
)

g = Grader()
def check(name, cond, detail=""):      # thin aliases over the shared grader
    return g.check(name, cond, detail)
def run_check(name, fn, expect_error=False):
    return g.run_check(name, fn, expect_error)

print("Python", platform.python_version())
print("Required prompt sections:", REQUIRED_SECTIONS)
print("Kit loaded. Grader ready.")

## Part A -- Scenario and data (given)

**Timberline Home Supply** (a fictional home-improvement retailer) needs a returns triage assistant. You will turn a vague prompt into a precise, testable one.

**Policy K-Doc (6 numbered lines)**

1. Return window: 30 calendar days from delivery date.
2. Items must be in resellable condition; all original accessories are required for a full refund. Missing accessories reduce to a partial refund.
3. Opened power tools are returnable within the window but carry a 15 percent restocking fee.
4. Custom-mixed paint and cut-to-length materials are non-returnable once prepared.
5. If the receipt is missing, offer account lookup by email plus the last four digits of the card.
6. If the return window has expired and the item has an active manufacturer warranty, provide the warranty repair contact.

**Customer messages**

- **C-101**: cordless drill, 12 days ago, works fine, box opened and used once.
- **C-102**: shelving unit from last week, receipt lost.
- **C-103**: custom-tinted paint, wrong color, wants a refund.
- **C-104**: patio heater delivered 45 days ago, igniter just failed.
- **C-105**: light fixture bought 10 days ago, mounting bracket thrown away.


In [ ]:
POLICY_TEXT = """1) Return window: 30 calendar days from delivery date.
2) Items must be in resellable condition; all original accessories are required for a full refund. Missing accessories reduce to a partial refund.
3) Opened power tools are returnable within the window but carry a 15 percent restocking fee.
4) Custom-mixed paint and cut-to-length materials are non-returnable once prepared.
5) If the receipt is missing, offer account lookup by email plus the last four digits of the card.
6) If the return window has expired and the item has an active manufacturer warranty, provide the warranty repair contact."""

INPUTS_TEXT = """C-101: I bought a cordless drill 12 days ago. It works fine but I want to return it. I did open the box and use it once.
C-102: I want to return a shelving unit that arrived last week, but I cannot find my receipt.
C-103: The custom-tinted paint I had mixed is the wrong color. I want a refund.
C-104: My patio heater was delivered 45 days ago and the igniter just failed.
C-105: I am returning a light fixture I bought 10 days ago, but I threw away the mounting bracket it came with."""

MESSAGES = [{"id": f"C-10{i}"} for i in range(1, 6)]
print(len(MESSAGES), "messages loaded")

## Part B -- Output contract as a Pydantic model  `TODO 1`

Model output is only useful if it obeys a contract. Encode the contract so malformed output is **rejected**, not silently accepted.

Build two models.

**`TriageRecord`** with fields:
- `input_id` : string shaped exactly `C-###`.
- `eligibility` : one of `eligible`, `ineligible`, `needs_lookup`.
- `action` : one of `refund`, `partial_refund`, `deny`, `warranty_repair`, `lookup`.
- `fee_percent` : a number or `null` (never a string such as `"15%"`).
- `reasons` : a list of strings, and it must be non-empty.
- `policy_lines` : a list of strings, each one of `"1"` through `"6"`.

**`TriageReport`** with fields:
- `records` : a list of `TriageRecord`.
- `stats` : an object that includes an integer `count`, where `count` equals the number of records.

Both models must **reject unknown fields**. Use the current Pydantic v2 idioms for the cohort stack (`pydantic==2.13.4`).


In [ ]:
from pydantic import BaseModel, ConfigDict, Field, field_validator
from typing import Optional


class TriageRecord(BaseModel):
    """One triage decision. See the contract in the markdown above.

    Reject unknown fields. Enforce the id shape, the two enums, the
    non-empty reasons rule, and the policy-line range.
    """
    # TODO: declare fields, validators, and model_config here.


class TriageReport(BaseModel):
    """The full report: a list of records plus a stats object whose
    integer count equals len(records). Reject unknown fields.
    """
    # TODO: declare fields, the count invariant, and model_config here.

In [ ]:
# ---- check TODO 1 (given) ----
GOOD_REC = {
    "records": [{
        "input_id": "C-101", "eligibility": "eligible", "action": "partial_refund",
        "fee_percent": 15, "reasons": ["opened power tool"], "policy_lines": ["1", "3"],
    }],
    "stats": {"count": 1},
}
def _accepts_good():
    r = TriageReport.model_validate(GOOD_REC)
    assert r.records[0].fee_percent == 15          # round-trips a real field
run_check("contract accepts a valid report", _accepts_good)

_bad = {
    "bad enum action":  {**GOOD_REC, "records": [{**GOOD_REC["records"][0], "action": "nope"}]},
    "string fee":       {**GOOD_REC, "records": [{**GOOD_REC["records"][0], "fee_percent": "15%"}]},
    "empty reasons":    {**GOOD_REC, "records": [{**GOOD_REC["records"][0], "reasons": []}]},
    "extra field":      {**GOOD_REC, "records": [{**GOOD_REC["records"][0], "note": "x"}]},
    "count mismatch":   {**GOOD_REC, "stats": {"count": 9}},
    "policy line 9":    {**GOOD_REC, "records": [{**GOOD_REC["records"][0], "policy_lines": ["9"]}]},
}
for label, payload in _bad.items():
    run_check(f"contract rejects: {label}",
              lambda p=payload: TriageReport.model_validate(p), expect_error=True)

## Part C -- Prompt structure and a linter  `TODO 2`

A prompt is code. Before it ships it should pass a **structural lint**: the required sections are present, non-empty, and in a sensible order. The canonical order is:

`ROLE, INSTRUCTION, CONTEXT, STEP_LIST, CONSTRAINTS, OUTPUT_CONTRACT, INPUTS, ACCEPTANCE_TESTS, SELF_VERIFY`

First, a quick demonstration of **why structure matters**, then you write the linter.


In [ ]:
# ---- why structure matters (given demonstration) ----
import json
bare_sections = {
    "ROLE": "You help with returns.",
    "INSTRUCTION": "Look at each message and decide what to do.",
    "CONTEXT": POLICY_TEXT,
    "INPUTS": INPUTS_TEXT,
}
bare_out = simulate_model(render_prompt(bare_sections), MESSAGES)
print("First line of the bare-prompt output:")
print("  ", repr(bare_out.splitlines()[0]))
try:
    json.loads(bare_out)
    print("Parsed as JSON.")
except json.JSONDecodeError as e:
    print("Strict json.loads FAILED:", e)
    print("Lesson: with no output contract, the model wraps JSON in prose.")

Now implement the linter.

Write `lint_prompt(text)` that returns a dict with keys:
- `missing` : required tags that are absent.
- `empty` : tags present but whose body is blank.
- `order` : the tags actually found, in the order they appear.
- `order_ok` : `True` when the found tags appear in canonical order.

A tag is an opening `<ROLE>` with a matching closing tag. Use `REQUIRED_SECTIONS` for the canonical list and order.


In [ ]:
import re

def lint_prompt(text: str) -> dict:
    """Return {missing, empty, order, order_ok} for a tagged prompt.

    missing:  required tags not found
    empty:    tags found but with a blank body
    order:    tags found, in the order they appear in the text
    order_ok: True when 'order' is the canonical order with missing tags skipped
    """
    raise NotImplementedError("Implement lint_prompt, then re-run.")

In [ ]:
# ---- check TODO 2 (given) ----
try:
    _wl = lint_prompt(render_prompt(bare_sections))
    check("linter flags missing sections", len(_wl["missing"]) > 0, str(_wl["missing"]))
    check("linter: bare prompt is missing OUTPUT_CONTRACT", "OUTPUT_CONTRACT" in _wl["missing"])
    check("linter: scrambled order is caught",
          lint_prompt("<INSTRUCTION>x</INSTRUCTION><ROLE>y</ROLE>")["order_ok"] is False)
    check("linter: empty body is caught", lint_prompt("<ROLE></ROLE>")["empty"] == ["ROLE"])
except NotImplementedError:
    check("TODO 2 lint_prompt implemented", False, "not implemented yet")
except Exception as e:
    check("TODO 2 lint_prompt runs without error", False, f"{type(e).__name__}: {e}")

## Part D -- Turn a weak prompt into a strong one  `TODO 3`

Below is a **weak prompt**: it asks for JSON, so the output parses, but it omits the field-level discipline. You will see it fail the acceptance tests in Part F.

Your job: build `STRONG_SECTIONS`, a dict keyed by the canonical tags, that a competent model can follow to produce fully contract-conformant output. Think about what the weak prompt is missing. At minimum your prompt should make the model:
- return `fee_percent` as a number, not a string,
- cite the policy line numbers it used,
- give a non-empty, policy-grounded reason for every record,
- return only the contracted fields, as strict JSON with no prose.

Include every canonical section. The provided `WEAK_SECTIONS` shows the format.


In [ ]:
# ---- the weak prompt (given) ----
WEAK_SECTIONS = {
    "ROLE": "You help with returns.",
    "INSTRUCTION": "Look at each message and decide what to do.",
    "CONTEXT": POLICY_TEXT,
    "OUTPUT_CONTRACT": "Return ONLY JSON with a records list and a stats object. No prose.",
    "INPUTS": INPUTS_TEXT,
}
print("Weak prompt renders", len(render_prompt(WEAK_SECTIONS)), "chars")

In [ ]:
# TODO 3: author a disciplined prompt. Fill every canonical section.
STRONG_SECTIONS = {
    # "ROLE": ...,
    # "INSTRUCTION": ...,
    # "CONTEXT": POLICY_TEXT,
    # "STEP_LIST": ...,
    # "CONSTRAINTS": ...,
    # "OUTPUT_CONTRACT": ...,
    # "INPUTS": INPUTS_TEXT,
    # "ACCEPTANCE_TESTS": ...,
    # "SELF_VERIFY": ...,
}

In [ ]:
# ---- check TODO 3 (given) ----
try:
    _lint = lint_prompt(render_prompt(STRONG_SECTIONS))
    check("strong prompt has no missing sections", _lint["missing"] == [], str(_lint["missing"]))
    check("strong prompt has no empty sections", _lint["empty"] == [], str(_lint["empty"]))
    check("strong prompt is in canonical order", _lint["order_ok"] is True)
except NotImplementedError:
    check("TODO 3 strong prompt built", False, "STRONG_SECTIONS not filled in")
except Exception as e:
    check("TODO 3 strong prompt lints", False, f"{type(e).__name__}: {e}")

def _strong_parses_and_validates():
    out = simulate_model(render_prompt(STRONG_SECTIONS), MESSAGES)
    TriageReport.model_validate(json.loads(out))
run_check("strong prompt output parses and validates", _strong_parses_and_validates)

## Part E -- Acceptance tests  `TODO 4`

Now write the suite that judges any raw model output. Implement `run_acceptance_tests(raw)` returning a list of `(name, passed, detail)` tuples. It must never raise.

Required checks:
1. `parses_as_json` -- `raw` parses with strict `json.loads`. If not, stop and return just this one failing result.
2. `matches_contract` -- the parsed object validates against `TriageReport`.
3. `C-101 opened power tool -> partial_refund fee 15` -- action is `partial_refund` and `fee_percent == 15`.
4. `C-102 missing receipt`: eligibility `needs_lookup` and action `lookup`.
5. `C-103 custom paint -> deny` -- eligibility `ineligible` and action `deny`.
6. `C-104 expired + warranty -> warranty_repair` -- action `warranty_repair` and `"6"` in its policy_lines.
7. `C-105 missing accessory -> partial_refund cites 2` -- action `partial_refund` and `"2"` in its policy_lines.
8. `all reasons non-empty` -- every record has a non-empty reasons list.
9. `all policy_lines cite 1..6 and non-empty` -- every record cites at least one line, all within 1 to 6.

Two fixtures are provided to test your suite in isolation.


In [ ]:
GOOD_OUTPUT = r"""{
  "records": [
    {
      "input_id": "C-101",
      "eligibility": "eligible",
      "action": "partial_refund",
      "fee_percent": 15,
      "reasons": [
        "Opened power tool within return window",
        "15 percent restocking fee applies"
      ],
      "policy_lines": [
        "1",
        "3"
      ]
    },
    {
      "input_id": "C-102",
      "eligibility": "needs_lookup",
      "action": "lookup",
      "fee_percent": null,
      "reasons": [
        "Receipt missing",
        "Offer lookup by email and last four card digits"
      ],
      "policy_lines": [
        "5"
      ]
    },
    {
      "input_id": "C-103",
      "eligibility": "ineligible",
      "action": "deny",
      "fee_percent": null,
      "reasons": [
        "Custom-mixed paint is non-returnable once prepared"
      ],
      "policy_lines": [
        "4"
      ]
    },
    {
      "input_id": "C-104",
      "eligibility": "ineligible",
      "action": "warranty_repair",
      "fee_percent": null,
      "reasons": [
        "Return window expired",
        "Item has an active manufacturer warranty"
      ],
      "policy_lines": [
        "1",
        "6"
      ]
    },
    {
      "input_id": "C-105",
      "eligibility": "eligible",
      "action": "partial_refund",
      "fee_percent": null,
      "reasons": [
        "Original accessory missing",
        "Full refund requires all accessories"
      ],
      "policy_lines": [
        "2"
      ]
    }
  ],
  "stats": {
    "count": 5
  }
}"""

BAD_OUTPUT = r"""{
  "records": [
    {
      "input_id": "C-101",
      "eligibility": "eligible",
      "action": "partial_refund",
      "fee_percent": "15%",
      "reasons": [],
      "policy_lines": []
    },
    {
      "input_id": "C-102",
      "eligibility": "needs_lookup",
      "action": "lookup",
      "fee_percent": null,
      "reasons": [],
      "policy_lines": []
    },
    {
      "input_id": "C-103",
      "eligibility": "ineligible",
      "action": "deny",
      "fee_percent": null,
      "reasons": [],
      "policy_lines": []
    },
    {
      "input_id": "C-104",
      "eligibility": "ineligible",
      "action": "warranty_repair",
      "fee_percent": null,
      "reasons": [],
      "policy_lines": []
    },
    {
      "input_id": "C-105",
      "eligibility": "eligible",
      "action": "partial_refund",
      "fee_percent": null,
      "reasons": [],
      "policy_lines": []
    }
  ],
  "stats": {
    "count": 5
  }
}"""
print("fixtures loaded:", len(GOOD_OUTPUT), len(BAD_OUTPUT), "chars")

In [ ]:
def run_acceptance_tests(raw: str) -> list:
    """Return a list of (name, passed, detail) tuples. Never raise.

    Implement the nine checks described in the markdown above, in order.
    If the output does not parse as JSON, return only the single failing
    'parses_as_json' result.
    """
    raise NotImplementedError("Implement run_acceptance_tests, then re-run.")

In [ ]:
# ---- check TODO 4 (given) ----
try:
    _good = run_acceptance_tests(GOOD_OUTPUT)
    check("suite passes the good fixture 9/9",
          sum(ok for _, ok, _ in _good) == 9, f"{sum(ok for _,ok,_ in _good)}/9")
    _bad = run_acceptance_tests(BAD_OUTPUT)
    _bad_fail = {n for n, ok, _ in _bad if not ok}
    check("suite catches the bad fixture contract violation", "matches_contract" in _bad_fail)
    check("suite catches the bad fixture empty reasons", "all reasons non-empty" in _bad_fail)
except NotImplementedError:
    check("TODO 4 run_acceptance_tests implemented", False, "not implemented yet")
except Exception as e:
    check("TODO 4 run_acceptance_tests runs", False, f"{type(e).__name__}: {e}")

## Part F -- The payoff: weak prompt vs strong prompt

Run the suite against both prompts' output. The weak prompt parses but violates the contract in several ways; the strong prompt passes everything. This is the loop you will run for real: prompt, output, test, repair.


In [ ]:
def board(label, sections):
    out = simulate_model(render_prompt(sections), MESSAGES)
    res = run_acceptance_tests(out)
    n = sum(ok for _, ok, _ in res)
    print()
    print(f"{label}: {n}/{len(res)} acceptance checks pass")
    for name, ok, detail in res:
        mark = "PASS" if ok else "FAIL"
        tail = (f"  -> {detail}" if not ok else "")
        print(f"   [{mark}] {name}{tail}")
    return n, len(res)

try:
    wp, wt = board("WEAK prompt", WEAK_SECTIONS)
    sp, st = board("STRONG prompt", STRONG_SECTIONS)
    check("weak prompt fails at least one acceptance check", wp < wt)
    check("strong prompt passes every acceptance check", sp == st)
except NotImplementedError:
    check("Part F runs (needs TODO 3 and TODO 4)", False, "finish TODO 3 and TODO 4 first")
except Exception as e:
    check("Part F runs", False, f"{type(e).__name__}: {e}")

## Part G -- Style and tone without breaking the contract  `TODO 5`

Tone is a real requirement, but it must not corrupt structure. Build two variants of the strong prompt by copying `STRONG_SECTIONS` and adding a single tone constraint to each, keeping every other section identical.

- `CONCISE_SECTIONS`: add a constraint that keeps each reason to eight words or fewer.
- `SUPPORTIVE_SECTIONS`: change the role to a warm, customer-friendly specialist and add a constraint that reasons stay factual with no apologies or promises.

Both must still lint clean and still pass all nine acceptance checks.


In [ ]:
# TODO 5: build two tone variants that keep the contract intact.
CONCISE_SECTIONS = {}      # copy STRONG_SECTIONS, add a brevity constraint
SUPPORTIVE_SECTIONS = {}   # copy STRONG_SECTIONS, warmer role + factual constraint

In [ ]:
# ---- check TODO 5 (given) ----
try:
    if not CONCISE_SECTIONS or not SUPPORTIVE_SECTIONS:
        raise NotImplementedError
    for _name, _secs in [("concise", CONCISE_SECTIONS), ("supportive", SUPPORTIVE_SECTIONS)]:
        _l = lint_prompt(render_prompt(_secs))
        check(f"{_name} variant lints clean", _l["missing"] == [] and _l["order_ok"])
        _r = run_acceptance_tests(simulate_model(render_prompt(_secs), MESSAGES))
        check(f"{_name} variant still passes all acceptance checks",
              all(ok for _, ok, _ in _r), str([n for n, ok, _ in _r if not ok]))
except NotImplementedError:
    check("TODO 5 tone variants built", False, "fill CONCISE_SECTIONS and SUPPORTIVE_SECTIONS")
except Exception as e:
    check("TODO 5 tone variants run", False, f"{type(e).__name__}: {e}")

## Stretch goals (optional)

**Stretch 1 -- Failure-driven repair.** Write `make_repair_block(results)` that reads the output of `run_acceptance_tests` and returns a `<REPAIR_NOTES>` block naming concrete fixes (numeric fee, grounded reasons, cited policy lines). Appending that block to the **weak** prompt should be enough to flip its output to fully conformant. This is the automated reason-to-fix loop.

**Stretch 2 -- Robust extraction, with a caveat.** Write `extract_json(raw)` that salvages a JSON object from a prose or fenced response, and show it can rescue the bare-prompt output. Note the lesson: a lenient parser papers over a bad prompt; the durable fix is the output contract, not a more forgiving client.


In [ ]:
def make_repair_block(results: list) -> str:
    """Return a <REPAIR_NOTES> block that names fixes for the failing checks.
    If nothing failed, say so. Appending this to a weak prompt should let a
    competent model correct its next output."""
    raise NotImplementedError("Stretch 1")

In [ ]:
# ---- check Stretch 1 (given) ----
try:
    _weak_prompt = render_prompt(WEAK_SECTIONS)
    _weak_res = run_acceptance_tests(simulate_model(_weak_prompt, MESSAGES))
    _repair = make_repair_block(_weak_res)
    _repaired = _weak_prompt + chr(10) * 2 + _repair
    _after = run_acceptance_tests(simulate_model(_repaired, MESSAGES))
    check("repair block flips the weak prompt to fully passing",
          all(ok for _, ok, _ in _after), str([n for n, ok, _ in _after if not ok]))
except NotImplementedError:
    check("Stretch 1 make_repair_block implemented", False, "not implemented yet")
except Exception as e:
    check("Stretch 1 make_repair_block runs", False, f"{type(e).__name__}: {e}")

In [ ]:
def extract_json(raw: str) -> dict:
    """Salvage a JSON object from a prose or fenced response. Raise if none."""
    raise NotImplementedError("Stretch 2")

In [ ]:
# ---- check Stretch 2 (given) ----
_bare = simulate_model(render_prompt(bare_sections), MESSAGES)
def _rescue():
    d = extract_json(_bare)
    assert d["stats"]["count"] == len(d["records"])
run_check("extract_json rescues the bare-prompt output", _rescue)

## Wrap-up

**Knowledge checks**
1. Which single section change moved the bare prompt from unparseable to parseable, and why?
2. The contract rejected a `fee_percent` of `"15%"`. Where should that be caught in a real pipeline: the prompt, the parser, or the schema? Argue for one.
3. Why keep acceptance tests separate from the prompt rather than trusting the SELF_VERIFY block alone?
4. Tone varied but scores held. What in the design made that possible?

**Responsible AI note.** The contract plus the acceptance suite are a lightweight output-safety gate: they refuse malformed or unpolicied output before it reaches a customer. In production you would log every rejection and route repeated failures to human review.


In [ ]:
print(g.summary())